# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. This biomedical dataset focuses on protein abundance, modifications, and sequences in human samples via mass spectrometry.

### Dataset Source
The dataset source is provided via a Croissant schema URL and consists of multiple record sets and fields, which will be referenced by their `@id` attributes.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant JSON-LD metadata schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset object
dataset = mlc.Dataset(croissant_url)

# Print some basic metadata (access via attribute)
print(f"Dataset: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(dataset.metadata, 'identifier', 'N/A')}")
print(f"License: {getattr(dataset.metadata, 'license', 'N/A')}")

## 2. Data Overview
List the available record sets in the dataset, and inspect their fields and columns by their `@id`. This information is essential for referencing when loading data or performing manipulations.

In [ ]:
# List all record sets with their @id and fields
print("Available record sets:")
for record_set in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id} | name: {getattr(record_set, 'name', None)}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print("  Fields and columns:")
        for field in record_set.fields:
            print(f"    Field @id: {field.id} | name: {getattr(field, 'name', None)} | dataType: {getattr(field, 'dataType', None)}")
            if hasattr(field, 'columns') and field.columns:
                for column in field.columns:
                    print(f"      Column @id: {column.id} | name: {getattr(column, 'name', None)}")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame. _Use only the exact `@id` values above._
You may need to iterate over all record sets or select one by its chosen `@id`.

In [ ]:
# Choose a record set by @id for extraction
# (Please replace these IDs if you customize; default example below tries all record sets)
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}

for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet {record_set_id}.")
            print(f"- Columns: {df.columns.tolist()}")
            print(df.head())
        else:
            print(f"RecordSet {record_set_id} yielded no data.")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")
# For further analysis, select a record set with actual tabular content
# Here we arbitrarily take the first loaded one
if dataframes:
    main_rs_id = list(dataframes)[0]
    print(f"Using RecordSet {main_rs_id} for analysis.")
else:
    main_rs_id = None

## 4. Exploratory Data Analysis (EDA)
Apply processing to the loaded data: filtering based on a numeric field, normalization, and grouping by category.
_Reference all fields using their exact `@id` as shown in the overview section above._

In [ ]:
import numpy as np

if main_rs_id is not None:
    # List available columns
    print(f"Columns in {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    
    # Guess/select a field likely numeric (e.g., Abundance, MW, pI); adapt @id as needed
    numeric_candidates = [col for col in dataframes[main_rs_id].columns if dataframes[main_rs_id][col].dtype in [np.float64, np.int64, float, int]]
    
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Choose any; adapt for specific use
        threshold = dataframes[main_rs_id][numeric_field].mean() if not np.isnan(dataframes[main_rs_id][numeric_field].mean()) else 0
        filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by a categorical field
        group_candidates = [col for col in dataframes[main_rs_id].columns if dataframes[main_rs_id][col].dtype == object]
        if group_candidates:
            group_field = group_candidates[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (mean aggregation):")
            print(grouped_df.head())
else:
    print("No record set with suitable data for EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and the effect of filtering/normalization. Plots will only show if the DataFrame is non-empty and an appropriate numeric column is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if main_rs_id is not None and numeric_candidates:
    plt.figure(figsize=(10, 5))
    sns.histplot(dataframes[main_rs_id][numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # Scatter plot if grouping available
    if group_candidates:
        gfield = group_candidates[0]
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=gfield, y=numeric_field, data=dataframes[main_rs_id])
        plt.title(f"{numeric_field} by {gfield}")
        plt.xlabel(gfield)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook showed how to load the FAIR² dataset using the [Croissant](https://mlcommons.org/croissant/) format via the `mlcroissant` library, inspect record sets and their field/column `@id`s, extract records into pandas DataFrames, and carry out basic exploratory data analysis and visualization.

- Ensure all references to fields and record sets use their `@id`.
- With the Croissant schema, you can systematically explore other record sets or fields.

**Key observations:**
- The dataset supports analysis of protein attributes by leveraging the structured metadata and record sets defined in Croissant.
- Filtering and normalization steps uncover protein abundance trends and effects of pre-processing.

For advanced analyses, consult the schema's documentation and the [mlcroissant documentation](https://mlcommons.github.io/croissant/).